# Routing ???????

?? notebook ???
- ???? `RoutingDataset` ????/????
- ?? `routing_model.py` ????`GR2N`?
- ???????loss???
- ????????????????????????

In [ ]:
from pathlib import Path
import sys
import json
import yaml
import time

import torch
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# ?????????? PYTHONPATH
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from datasets import build_train_val_dataloaders
from models import build_model
from trainers import Trainer, build_loss, build_optimizer, build_scheduler, set_seed

In [ ]:
TRAIN_CFG_PATH = PROJECT_ROOT / "configs" / "train.yaml"
DATA_CFG_PATH = PROJECT_ROOT / "configs" / "data.yaml"
MODEL_CFG_PATH = PROJECT_ROOT / "configs" / "model.yaml"

def load_yaml(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def merge_model_cfg(model_cfg_file: dict, train_cfg: dict):
    cfg = {}
    if isinstance(model_cfg_file.get("model"), dict):
        cfg.update(model_cfg_file["model"])
    # ?? train.yaml ?????
    if isinstance(train_cfg.get("model"), dict):
        cfg.update(train_cfg["model"])
    return cfg

train_cfg = load_yaml(TRAIN_CFG_PATH)
model_cfg_file = load_yaml(MODEL_CFG_PATH)
model_cfg = merge_model_cfg(model_cfg_file, train_cfg)

# ===== ????????? =====
quick_run = False
if quick_run:
    train_cfg.setdefault("trainer", {})["epochs"] = 2
    train_cfg.setdefault("dataloader", {}).setdefault("train", {})["batch_size"] = 64
    train_cfg.setdefault("dataloader", {}).setdefault("val", {})["batch_size"] = 64

print("model_cfg:")
print(json.dumps(model_cfg, ensure_ascii=False, indent=2))

In [ ]:
seed = int(train_cfg.get("seed", 42))
set_seed(seed)

dataset_cfg = dict(train_cfg.get("dataset", {}))
dataset_cfg.setdefault("data_cfg_path", str(DATA_CFG_PATH))
dataset_cfg.setdefault("model_cfg_path", str(MODEL_CFG_PATH))

train_loader_cfg = dict(train_cfg.get("dataloader", {}).get("train", {}))
val_loader_cfg = dict(train_cfg.get("dataloader", {}).get("val", {}))

train_ds, val_ds, train_loader, val_loader = build_train_val_dataloaders(
    dataset_kwargs=dataset_cfg,
    train_loader_kwargs=train_loader_cfg,
    val_loader_kwargs=val_loader_cfg,
)

print("Train runtime schema:")
print(train_ds.format_runtime_schema())
print("Val runtime schema:")
print(val_ds.format_runtime_schema())

In [ ]:
if "pred_len" not in model_cfg and "n_pred" in dataset_cfg:
    model_cfg["pred_len"] = dataset_cfg["n_pred"]

model = build_model(model_cfg=model_cfg, dataset=train_ds)
criterion = build_loss(train_cfg.get("loss", {}))
optimizer = build_optimizer(model.parameters(), train_cfg.get("optimizer", {}))

trainer_cfg = dict(train_cfg.get("trainer", {}))
epochs = int(trainer_cfg.get("epochs", 20))
scheduler = build_scheduler(
    optimizer=optimizer,
    scheduler_cfg=train_cfg.get("scheduler", {}),
    total_epochs=epochs,
    steps_per_epoch=len(train_loader),
)

checkpoint_dir = trainer_cfg.get("checkpoint_dir", "checkpoints/routing_notebook")
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=train_cfg.get("device", "auto"),
    use_amp=bool(trainer_cfg.get("use_amp", False)),
    amp_dtype=str(trainer_cfg.get("amp_dtype", "float16")),
    grad_clip_norm=trainer_cfg.get("grad_clip_norm", None),
    log_interval=int(trainer_cfg.get("log_interval", 50)),
    checkpoint_dir=str(PROJECT_ROOT / checkpoint_dir),
    monitor=str(trainer_cfg.get("monitor", "val_loss")),
    monitor_mode=str(trainer_cfg.get("monitor_mode", "min")),
    early_stopping_patience=trainer_cfg.get("early_stopping_patience", None),
    keep_last_k=int(trainer_cfg.get("keep_last_k", 3)),
)

print("Model:", type(model).__name__)
print("Checkpoint dir:", trainer.checkpoint_dir)

In [ ]:
# ????
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=epochs,
    resume_path=trainer_cfg.get("resume_path", None),
)

print("best_epoch:", trainer.best_epoch)
print("best_metric:", trainer.best_metric)
print("best_ckpt:", trainer.get_best_checkpoint_path())

In [ ]:
# ??????
hist_df = pd.read_csv(trainer.history_csv_path)
display(hist_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
if "val_loss" in hist_df.columns:
    axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val_loss")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("epoch")
axes[0].legend()

if "val_nse" in hist_df.columns:
    axes[1].plot(hist_df["epoch"], hist_df["val_nse"], label="val_nse")
axes[1].set_title("NSE Curve")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ???????? + ??????
def compute_metrics(pred: torch.Tensor, target: torch.Tensor):
    err = pred - target
    mse = torch.mean(err * err)
    mae = torch.mean(torch.abs(err))
    rmse = torch.sqrt(mse + 1e-12)
    denom = torch.sum((target - torch.mean(target)) ** 2)
    nse = 1.0 - torch.sum(err ** 2) / (denom + 1e-12)
    return {
        "loss": float(mse.detach().cpu()),
        "mae": float(mae.detach().cpu()),
        "rmse": float(rmse.detach().cpu()),
        "nse": float(nse.detach().cpu()),
    }

pred_norm, target_norm = trainer.predict(val_loader, return_target=True)
metrics_norm = compute_metrics(pred_norm, target_norm)

pred_denorm = val_ds.inverse_transform_streamflow_tensor(pred_norm)
target_denorm = val_ds.inverse_transform_streamflow_tensor(target_norm)
metrics_denorm = compute_metrics(pred_denorm, target_denorm)

print("metrics_norm:", json.dumps(metrics_norm, ensure_ascii=False, indent=2))
print("metrics_denorm:", json.dumps(metrics_denorm, ensure_ascii=False, indent=2))

In [ ]:
# ?????????????????
max_points = 400
num_outlets = pred_denorm.shape[-1]
names = getattr(val_ds, "outlet_names", [f"outlet_{i}" for i in range(num_outlets)])

for i in range(num_outlets):
    p = pred_denorm[:, 0, i].detach().cpu().numpy()[:max_points]
    t = target_denorm[:, 0, i].detach().cpu().numpy()[:max_points]

    plt.figure(figsize=(10, 3))
    plt.plot(t, label="obs", linewidth=1.5)
    plt.plot(p, label="pred", linewidth=1.2)
    plt.title(f"Outlet: {names[i]}")
    plt.xlabel("time index")
    plt.ylabel("streamflow (denorm)")
    plt.legend()
    plt.tight_layout()
    plt.show()